# Avatar falante — caminho 100% gratuito

Este notebook roda no **Google Colab (GPU T4 gratuita)** e faz as duas partes
que a sua máquina/servidor não consegue fazer sem GPU:

1. **Voz clonada** com [Chatterbox Multilingual pt-br](https://huggingface.co/ResembleAI/Chatterbox-Multilingual-pt-br) — licença **MIT** (uso comercial liberado).
2. **Lipsync** com [LatentSync](https://github.com/bytedance/LatentSync) da ByteDance — licença **Apache 2.0** (uso comercial liberado).

O resultado é um `avatar.mp4` com você falando o texto. Baixe esse arquivo e
coloque em `vsl/entrada/avatar/` para o motor de montagem usar.

---

### O que você precisa ter em mãos

| Arquivo | O que é | Como gravar |
|---|---|---|
| `base.mp4` | Você olhando pra câmera, 30–90s, **boca aparecendo bem**, sem cortes | Celular na horizontal, luz na frente do rosto, fundo neutro |
| `voz.wav` | 10–20s da sua voz limpa, sem ruído/música | Fale um parágrafo qualquer no gravador do celular |

### Limites honestos do plano gratuito do Colab
- A GPU T4 (15 GB) roda o **LatentSync 1.5** (8 GB). A versão 1.6 pede 18 GB e **não cabe** no plano grátis.
- A sessão cai depois de algumas horas e a GPU não é garantida em horário de pico.
- Dá pra produzir alguns vídeos por dia com tranquilidade. Não serve pra fabricar 50 por hora.

> ⚠️ Este notebook foi escrito a partir da documentação oficial dos dois projetos,
> mas **não foi executado** no ambiente onde foi criado (um container sem GPU).
> A primeira rodada pode pedir um ajuste de versão de dependência — é normal
> em projetos de pesquisa. Se travar, o caminho alternativo está no final.

## 1. Confirmar que a GPU está ligada

No menu: **Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("CUDA disponível:", torch.cuda.is_available())

## 2. Enviar seus arquivos

Sobe o vídeo base e a amostra da sua voz.

In [ ]:
import os
os.makedirs('/content/meus', exist_ok=True)
%cd /content/meus
from google.colab import files
print("Selecione base.mp4 e voz.wav (pode mandar os dois de uma vez):")
enviados = files.upload()
print("\nRecebidos:", list(enviados))

## 3. Instalar o Chatterbox (voz clonada, MIT)

In [ ]:
!pip install -q chatterbox-tts
print("ok")

## 4. Gerar a narração com a SUA voz

Cole o roteiro abaixo. Uma cena por linha — o nome do arquivo de cada cena
(`cena_01.wav`, `cena_02.wav`...) é o que você vai referenciar no roteiro JSON
do motor de montagem, no campo `narracao`.

In [ ]:
ROTEIRO = """
Voce trabalhou a vida toda e o INSS negou o seu beneficio?
Nove de cada dez negativas acontecem por erro no pedido, nao por falta de direito.
Meu nome e Faria. Sou advogado e ja revisei centenas de processos como o seu.
Em quarenta e oito horas eu analiso o seu caso e digo exatamente o que deu errado.
Chama no WhatsApp agora. A primeira analise e gratuita.
"""

VOZ_REFERENCIA = "/content/meus/voz.wav"   # 10-20s da sua voz

import os, torch, torchaudio
from chatterbox.mtl_tts import ChatterboxMultilingualTTS

modelo = ChatterboxMultilingualTTS.from_pretrained(device="cuda")
os.makedirs("/content/narracao", exist_ok=True)

linhas = [l.strip() for l in ROTEIRO.strip().splitlines() if l.strip()]
for i, linha in enumerate(linhas, 1):
    wav = modelo.generate(linha, language_id="pt", audio_prompt_path=VOZ_REFERENCIA)
    destino = f"/content/narracao/cena_{i:02d}.wav"
    torchaudio.save(destino, wav, modelo.sr)
    print(f"[{i}/{len(linhas)}] {destino}  <-  {linha[:60]}")

# junta tudo num arquivo so, que e o que o lipsync consome
!cd /content/narracao && ls cena_*.wav | sed "s/^/file '/;s/$/'/" > lista.txt && \
  ffmpeg -y -v error -f concat -safe 0 -i lista.txt -ar 16000 -ac 1 /content/narracao/completa.wav
print("\nNarracao completa: /content/narracao/completa.wav")

Ouça antes de seguir. Se a voz não ficou parecida, troque a amostra de referência por uma gravação mais limpa e rode de novo.

In [ ]:
from IPython.display import Audio
Audio("/content/narracao/completa.wav")

## 5. Instalar o LatentSync (lipsync, Apache 2.0)

Baixa o código e os pesos. São alguns GB — leva uns minutos.

In [ ]:
%cd /content
!git clone -q https://github.com/bytedance/LatentSync.git
%cd /content/LatentSync
!apt-get -qq install -y libgl1 > /dev/null
!pip install -q -r requirements.txt
!pip install -q huggingface_hub

# LatentSync 1.5 = 8 GB de VRAM, cabe na T4 gratuita.
# (a 1.6 e melhor, mas pede 18 GB e so roda em GPU maior)
!huggingface-cli download ByteDance/LatentSync-1.5 whisper/tiny.pt --local-dir checkpoints
!huggingface-cli download ByteDance/LatentSync-1.5 latentsync_unet.pt --local-dir checkpoints
!ls -la checkpoints

## 6. Rodar o lipsync

Gruda a narração na sua boca no vídeo base.

- `inference_steps`: 20 é o equilíbrio. Até 50 melhora a imagem e demora mais.
- `guidance_scale`: 1.5 é o padrão. Mais alto gruda melhor nos lábios, mas pode tremer.

In [ ]:
%cd /content/LatentSync
!python -m scripts.inference \
    --unet_config_path "configs/unet/stage2.yaml" \
    --inference_ckpt_path "checkpoints/latentsync_unet.pt" \
    --inference_steps 20 \
    --guidance_scale 1.5 \
    --enable_deepcache \
    --video_path "/content/meus/base.mp4" \
    --audio_path "/content/narracao/completa.wav" \
    --video_out_path "/content/avatar.mp4"
print("pronto")

## 7. Conferir e baixar

In [ ]:
from IPython.display import HTML
from base64 import b64encode
dados = b64encode(open('/content/avatar.mp4','rb').read()).decode()
HTML(f'<video width=480 controls src="data:video/mp4;base64,{dados}">')

In [ ]:
from google.colab import files
files.download('/content/avatar.mp4')
# tambem vale baixar as narracoes separadas por cena:
!cd /content && zip -qr narracao.zip narracao
files.download('/content/narracao.zip')

---

## Deu errado? Alternativas que também não custam nada

| Situação | O que fazer |
|---|---|
| Conflito de dependência no LatentSync | Use o **Space gratuito** do próprio projeto: https://huggingface.co/spaces/fffiloni/LatentSync (sobe vídeo + áudio pelo navegador, sem instalar nada) |
| Colab sem GPU disponível | Tente mais tarde, ou use o Space acima |
| Voz clonada não convenceu | **Grave sua própria voz lendo o roteiro.** É grátis, é literalmente a sua voz, e para VSL costuma converter melhor que qualquer clone |
| Quer pular o lipsync de vez | Grave 2–3 min seus falando e use o motor de montagem só com cortes, legendas e B-roll |

## Depois daqui

Coloque o `avatar.mp4` em `vsl/entrada/avatar/` e as narrações em
`vsl/entrada/narracao/`, aponte no roteiro JSON e rode:

```bash
python3 vsl/bin/montar.py meu-roteiro.json --formato todos
```